# Proyecto Integrador de Minería de Datos I  
## Notebook 02 — Calidad, limpieza y preparación de datos

**Carrera:** Tecnicatura Superior en Ciencia de Datos e Inteligencia Artificial  
**Asignatura:** Minería de Datos I  
**Dataset:** `streaming_users_dirty.json`  
**Integrantes:** Thir Ferreyra Nadia Lorena y Constantinidi Leandro Exequiel  
**Comisión:** Turno Tarde  
**Fecha de corte utilizada:** 27 de junio de 2026  

---

## Propósito de esta etapa

En este notebook se aplican las decisiones de limpieza que fueron anticipadas en la inspección inicial.

Cada tratamiento se presenta con la misma secuencia:

> **Evidencia observada → decisión aplicada → impacto comprobado**

El objetivo no es conseguir un dataset con cero valores faltantes a cualquier costo. El objetivo es obtener un conjunto de datos coherente, reproducible y adecuado para los análisis posteriores, evitando eliminar o inventar información sin justificación.

### Resultados que se generarán

- Dataset procesado: `data/processed/streaming_users_clean.csv`
- Registro de transformaciones: `logs/pipeline_log.csv`


## 1. Importación de librerías

Se utilizan:

- `pandas` para cargar, transformar y validar los datos;
- `numpy` para representar valores faltantes;
- `pathlib` para trabajar con rutas de forma reproducible;
- `re` para reconocer formatos de fecha mediante patrones.


In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Carga del dataset original

El archivo se lee directamente desde `data/raw/`. Esa carpeta no será modificada.

Se crea:

- `df_raw`: copia de control del archivo original;
- `df`: DataFrame sobre el que se aplicará el proceso de limpieza.


In [2]:
rutas_posibles = [
    Path("../data/raw/streaming_users_dirty.json"),
    Path("data/raw/streaming_users_dirty.json"),
    Path("/content/PI_Mineria_Datos_1/data/raw/streaming_users_dirty.json"),
]

ruta_datos = next((ruta for ruta in rutas_posibles if ruta.exists()), None)

if ruta_datos is None:
    raise FileNotFoundError(
        "No se encontró streaming_users_dirty.json en data/raw/."
    )

df_raw = pd.read_json(ruta_datos)
df = df_raw.copy(deep=True)

filas_originales = len(df_raw)

print(f"Archivo cargado desde: {ruta_datos.resolve()}")
print(f"Dimensiones iniciales: {df.shape}")
display(df.head())

Archivo cargado desde: /mnt/data/PI_Mineria_Datos_1_GRUPAL/data/raw/streaming_users_dirty.json
Dimensiones iniciales: (8160, 8)


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.80,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,"1,173.40",Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.00,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.40,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.80,Perú,Thriller,2020-09-30,1


## 3. Función para auditar el pipeline

La consigna exige registrar las transformaciones relevantes en `pipeline_log.csv`.

La función `auditar()` guarda:

- número de paso;
- descripción de la transformación;
- cantidad de filas;
- cantidad total de valores nulos;
- porcentaje de filas retenidas respecto del dataset original.

La retención se calcula sobre filas porque las columnas originales se conservarán durante toda esta etapa.


In [3]:
pipeline_log = []

def auditar(dataframe, paso, descripcion):
    pipeline_log.append({
        "Paso": paso,
        "Descripción": descripcion,
        "Filas": len(dataframe),
        "Nulos": int(dataframe.isna().sum().sum()),
        "Retención (%)": round(len(dataframe) / filas_originales * 100, 2),
    })

auditar(df, 0, "Dataset original")

display(pd.DataFrame(pipeline_log))

,Paso,Descripción,Filas,Nulos,Retención (%)
0,0,Dataset original,8160,753,100.00


# 4. Tratamiento de identificadores duplicados

## 4.1 Evidencia observada

La inspección inicial mostró:

- 8.160 filas;
- 8.000 identificadores diferentes;
- 160 repeticiones adicionales de `user_id`;
- 126 repeticiones completamente idénticas;
- 34 repeticiones con alguna diferencia.

Antes de eliminar registros se analiza la estructura del archivo para determinar cuál aparición corresponde conservar.


In [4]:
bloque_principal = df.iloc[:8000]
bloque_agregado = df.iloc[8000:]

estructura_esperada = np.arange(10000, 18000)

evidencia_duplicados = pd.DataFrame({
    "comprobacion": [
        "Filas del bloque principal",
        "IDs únicos en el bloque principal",
        "Primer ID del bloque principal",
        "Último ID del bloque principal",
        "IDs consecutivos 10000-17999",
        "Filas agregadas al final",
        "IDs finales ya presentes en el bloque principal",
    ],
    "resultado": [
        len(bloque_principal),
        bloque_principal["user_id"].nunique(),
        bloque_principal["user_id"].min(),
        bloque_principal["user_id"].max(),
        np.array_equal(
            bloque_principal["user_id"].to_numpy(),
            estructura_esperada,
        ),
        len(bloque_agregado),
        bloque_agregado["user_id"].isin(
            bloque_principal["user_id"]
        ).all(),
    ],
})

display(evidencia_duplicados)

,comprobacion,resultado
0,Filas del bloque principal,8000
1,IDs únicos en el bloque principal,8000
2,Primer ID del bloque principal,10000
3,Último ID del bloque principal,17999
4,IDs consecutivos 10000-17999,True
5,Filas agregadas al final,160
6,IDs finales ya presentes en el bloque principal,True


### Interpretación

Las primeras 8.000 filas contienen una secuencia completa, ordenada y sin repeticiones desde el usuario 10000 hasta el 17999.

Las 160 filas restantes aparecen al final y todos sus identificadores ya existían en el bloque principal. Esta estructura constituye evidencia de que fueron agregadas posteriormente como copias.

Por ello, el criterio elegido es:

> **Conservar la primera aparición de cada `user_id` y eliminar las apariciones posteriores.**

No se elige la última versión porque el dataset no incluye una fecha de actualización del registro que permita afirmar que sea más reciente o más confiable.


In [5]:
duplicados_exactos_antes = int(df.duplicated().sum())
repeticiones_id_antes = int(df.duplicated(subset="user_id").sum())

# Comparar las dos apariciones para documentar cuántas tenían diferencias.
comparacion_duplicados = []

for user_id, grupo in (
    df[df.duplicated(subset="user_id", keep=False)]
    .groupby("user_id")
):
    grupo = grupo.sort_index()
    primera = grupo.iloc[0]
    segunda = grupo.iloc[1]
    columnas_diferentes = []

    for columna in df.columns:
        valor_1 = primera[columna]
        valor_2 = segunda[columna]

        iguales = (
            (pd.isna(valor_1) and pd.isna(valor_2))
            or valor_1 == valor_2
        )

        if not iguales:
            columnas_diferentes.append(columna)

    comparacion_duplicados.append({
        "user_id": user_id,
        "cantidad_diferencias": len(columnas_diferentes),
        "columnas_diferentes": (
            ", ".join(columnas_diferentes)
            if columnas_diferentes
            else "Ninguna: copia exacta"
        ),
    })

comparacion_duplicados = pd.DataFrame(comparacion_duplicados)

display(
    comparacion_duplicados["cantidad_diferencias"]
    .value_counts()
    .sort_index()
    .rename_axis("cantidad_columnas_diferentes")
    .reset_index(name="cantidad_usuarios")
)

,cantidad_columnas_diferentes,cantidad_usuarios
0,0,126
1,1,32
2,2,2


## 4.2 Acción aplicada


In [6]:
filas_antes = len(df)

df = df.drop_duplicates(subset="user_id", keep="first").copy()

filas_eliminadas = filas_antes - len(df)

auditar(
    df,
    1,
    "Conservar la primera aparición de cada user_id",
)

print(f"Filas eliminadas: {filas_eliminadas}")
print(f"Filas restantes: {len(df)}")
print(f"user_id duplicados restantes: {df['user_id'].duplicated().sum()}")

Filas eliminadas: 160
Filas restantes: 8000
user_id duplicados restantes: 0


## 4.3 Impacto

Se eliminaron exactamente 160 apariciones posteriores y se conservaron los 8.000 usuarios originales.

La retención de filas es del 98,04 %. La pérdida está concentrada exclusivamente en registros duplicados documentados, no en usuarios únicos.


# 5. Estandarización de variables categóricas

## 5.1 Evidencia observada

Las variables `subscription_plan`, `country` y `favorite_genre` contienen categorías equivalentes escritas con:

- mayúsculas y minúsculas diferentes;
- tildes omitidas;
- abreviaturas;
- palabras en inglés;
- espacios finales;
- errores tipográficos.

Ejemplos:

- `Básico`, `basico`, `BASICO`, `Basic`;
- `Brasil`, `brasil`, `Brazil`, `BRA`;
- `Acción`, `ACCIÓN`, `accion`, `Action`;
- `Premium` y `Premiun`.

Estas diferencias fragmentan artificialmente una misma categoría.


In [7]:
categorias_antes = pd.DataFrame({
    "variable": [
        "subscription_plan",
        "country",
        "favorite_genre",
    ],
    "categorias_distintas_incluyendo_nulos": [
        df["subscription_plan"].nunique(dropna=False),
        df["country"].nunique(dropna=False),
        df["favorite_genre"].nunique(dropna=False),
    ],
})

display(categorias_antes)

for columna in [
    "subscription_plan",
    "country",
    "favorite_genre",
]:
    print(f"\nFrecuencias originales de {columna}:")
    display(
        df[columna]
        .value_counts(dropna=False)
        .rename_axis(columna)
        .reset_index(name="cantidad")
    )

,variable,categorias_distintas_incluyendo_nulos
0,subscription_plan,15
1,country,26
2,favorite_genre,29



Frecuencias originales de subscription_plan:


,subscription_plan,cantidad
0,Básico,3386
1,Estándar,2653
2,Premium,1481
3,basico,60
4,BASICO,52
5,Basic,52
6,básico,50
7,Std,48
8,Estándar,46
9,estandar,36



Frecuencias originales de country:


,country,cantidad
0,Chile,1115
1,Brasil,1107
2,Uruguay,1102
3,México,1100
4,Colombia,1096
5,Perú,1091
6,Argentina,1069
7,colombia,27
8,méxico,25
9,uruguay,24



Frecuencias originales de favorite_genre:


,favorite_genre,cantidad
0,Comedia,1091
1,Drama,1082
2,Thriller,1072
3,Romance,1067
4,Documental,1066
5,Acción,1059
6,Crime,1044
7,None,240
8,Action,22
9,COMEDIA,19


## 5.2 Decisión

Se construyen diccionarios explícitos de equivalencias.

Esta alternativa se prefiere a aplicar solamente `.title()` o `.lower()`, porque cambiar mayúsculas no resolvería abreviaturas, traducciones ni errores tipográficos.

En el género `Crime` se adopta la etiqueta española **Crimen** para mantener el idioma utilizado en las demás categorías.


In [8]:
mapa_plan = {
    "Básico": "Básico",
    "basico": "Básico",
    "BASICO": "Básico",
    "Basic": "Básico",
    "básico": "Básico",
    "Estándar": "Estándar",
    "Estándar ": "Estándar",
    "estandar": "Estándar",
    "STANDARD": "Estándar",
    "Std": "Estándar",
    "Premium": "Premium",
    "Premium ": "Premium",
    "PREMIUM": "Premium",
    "premium": "Premium",
    "Premiun": "Premium",
}

mapa_pais = {
    "Argentina": "Argentina",
    "Argentina ": "Argentina",
    "argentina": "Argentina",
    "ARG": "Argentina",
    "Brasil": "Brasil",
    "brasil": "Brasil",
    "Brazil": "Brasil",
    "BRA": "Brasil",
    "Chile": "Chile",
    "Chile ": "Chile",
    "chile": "Chile",
    "CHL": "Chile",
    "Colombia": "Colombia",
    "colombia": "Colombia",
    "COL": "Colombia",
    "México": "México",
    "méxico": "México",
    "Mexico": "México",
    "MEX": "México",
    "Perú": "Perú",
    "perú": "Perú",
    "Peru": "Perú",
    "PER": "Perú",
    "Uruguay": "Uruguay",
    "uruguay": "Uruguay",
    "URY": "Uruguay",
}

mapa_genero = {
    "Acción": "Acción",
    "ACCIÓN": "Acción",
    "accion": "Acción",
    "Action": "Acción",
    "Comedia": "Comedia",
    "Comedia ": "Comedia",
    "COMEDIA": "Comedia",
    "comedy": "Comedia",
    "Crime": "Crimen",
    "CRIME": "Crimen",
    "crime": "Crimen",
    "Crimen": "Crimen",
    "Documental": "Documental",
    "documental": "Documental",
    "Documentary": "Documental",
    "DOC": "Documental",
    "Drama": "Drama",
    "Drama ": "Drama",
    "DRAMA": "Drama",
    "drama": "Drama",
    "Romance": "Romance",
    "Romance ": "Romance",
    "ROMANCE": "Romance",
    "romance": "Romance",
    "Thriller": "Thriller",
    "Thriller ": "Thriller",
    "THRILLER": "Thriller",
    "thriler": "Thriller",
}

df["subscription_plan"] = df["subscription_plan"].map(mapa_plan)
df["country"] = df["country"].map(mapa_pais)
df["favorite_genre"] = df["favorite_genre"].map(mapa_genero)

auditar(
    df,
    2,
    "Estandarizar categorías de plan, país y género",
)

print("Categorías luego de la estandarización:")
for columna in [
    "subscription_plan",
    "country",
    "favorite_genre",
]:
    print(f"\n{columna}:")
    display(
        df[columna]
        .value_counts(dropna=False)
        .rename_axis(columna)
        .reset_index(name="cantidad")
    )

Categorías luego de la estandarización:

subscription_plan:


,subscription_plan,cantidad
0,Básico,3600
1,Estándar,2817
2,Premium,1583



country:


,country,cantidad
0,Chile,1164
1,México,1156
2,Brasil,1156
3,Uruguay,1143
4,Colombia,1142
5,Perú,1134
6,Argentina,1105



favorite_genre:


,favorite_genre,cantidad
0,Comedia,1137
1,Drama,1115
2,Romance,1110
3,Documental,1107
4,Thriller,1104
5,Acción,1103
6,Crimen,1084
7,NaN,240


## 5.3 Impacto

La estandarización reduce:

- planes: de 15 representaciones a 3 categorías reales;
- países: de 26 representaciones a 7 países;
- géneros: de 29 representaciones a 7 géneros más los faltantes.

No se eliminaron filas y no se inventaron categorías nuevas. Se unificaron distintas representaciones del mismo concepto.


# 6. Tratamiento de la edad

## 6.1 Evidencia observada

La distribución contiene valores aislados de -5, 0, 4, 130 y 150 años. No existen registros entre 5 y 12 años, mientras que desde los 13 años comienza una distribución continua que llega hasta los 80 años.

Por esa discontinuidad se adopta como rango analíticamente consistente **13 a 100 años**. El criterio no pretende establecer quién puede usar una plataforma de streaming; se limita a identificar valores incompatibles con el patrón interno de este dataset. Las edades altas plausibles, como 70 u 80 años, se conservan.

In [9]:
frecuencia_edades_extremas = (
    df.loc[
        (df["age"] < 13) | (df["age"] > 100),
        "age",
    ]
    .value_counts()
    .sort_index()
    .rename_axis("edad")
    .reset_index(name="cantidad")
)

print("Edades fuera del rango preliminar:")
display(frecuencia_edades_extremas)

medianas_edad_por_plan = (
    df.loc[df["age"].between(13, 100)]
    .groupby("subscription_plan")["age"]
    .agg(["count", "median", "mean"])
    .round(2)
)

print("\nEdad válida por plan:")
display(medianas_edad_por_plan)

Edades fuera del rango preliminar:


,edad,cantidad
0,-5,21
1,0,24
2,4,22
3,130,34
4,150,19



Edad válida por plan:


,count,median,mean
subscription_plan,,,
Básico,3547,33.00,33.67
Estándar,2769,34.00,33.82
Premium,1564,33.00,33.61


### Criterio de imputación

Las medianas por plan son prácticamente iguales: 33, 34 y 33 años.

No existe evidencia suficiente para justificar una imputación diferente por plan. Por eso se utiliza la **mediana global de las edades válidas**.

Se prefiere la mediana a la media porque es más resistente a valores extremos.


In [10]:
mascara_edad_invalida = ~df["age"].between(13, 100)
cantidad_edades_invalidas = int(mascara_edad_invalida.sum())

df.loc[mascara_edad_invalida, "age"] = np.nan

auditar(
    df,
    3,
    "Marcar edades fuera del rango 13-100 como faltantes",
)

mediana_edad = df["age"].median()

df["age"] = df["age"].fillna(mediana_edad).astype(int)

auditar(
    df,
    4,
    f"Imputar edad con mediana global ({mediana_edad:.0f})",
)

print(f"Edades inválidas tratadas: {cantidad_edades_invalidas}")
print(f"Mediana utilizada: {mediana_edad:.0f}")
print(f"Edades faltantes finales: {df['age'].isna().sum()}")
print(f"Rango final: {df['age'].min()} a {df['age'].max()} años")

Edades inválidas tratadas: 120
Mediana utilizada: 33
Edades faltantes finales: 0
Rango final: 13 a 80 años


## 6.2 Impacto

Se corrigieron 120 edades imposibles o incompatibles con la estructura observada.

No se eliminaron usuarios. La mediana aplicada fue 33 años y el rango final quedó entre 13 y 80 años.


# 7. Tratamiento del tiempo mensual de visualización

## 7.1 Evidencia observada

La variable presenta:

- valores faltantes;
- −1 y −120 minutos;
- 50.000 y 99.999 minutos;
- valores iguales a cero;
- valores altos, pero físicamente posibles.

Un mes de 31 días contiene como máximo:

`31 × 24 × 60 = 44.640 minutos`

Por lo tanto:

- valores negativos son imposibles;
- valores superiores a 44.640 también son imposibles;
- cero se conserva porque puede representar un usuario que no reprodujo contenido durante el mes.


In [11]:
minutos_mes_31_dias = 31 * 24 * 60

mascara_tiempo_imposible = (
    (df["monthly_watch_time_mins"] < 0)
    | (df["monthly_watch_time_mins"] > minutos_mes_31_dias)
)

resumen_tiempo_inicial = pd.DataFrame({
    "situacion": [
        "Faltantes originales después de quitar duplicados",
        "Valores negativos",
        "Valores superiores a 44.640",
        "Valores iguales a cero",
    ],
    "cantidad": [
        int(df["monthly_watch_time_mins"].isna().sum()),
        int((df["monthly_watch_time_mins"] < 0).sum()),
        int(
            (
                df["monthly_watch_time_mins"]
                > minutos_mes_31_dias
            ).sum()
        ),
        int((df["monthly_watch_time_mins"] == 0).sum()),
    ],
})

display(resumen_tiempo_inicial)

print("\nValores físicamente imposibles:")
display(
    df.loc[
        mascara_tiempo_imposible,
        "monthly_watch_time_mins",
    ]
    .value_counts()
    .sort_index()
    .rename_axis("minutos")
    .reset_index(name="cantidad")
)

,situacion,cantidad
0,Faltantes originales después de quitar duplicados,193
1,Valores negativos,49
2,Valores superiores a 44.640,31
3,Valores iguales a cero,164



Valores físicamente imposibles:


,minutos,cantidad
0,-120.00,29
1,-1.00,20
2,"50,000.00",11
3,"99,999.00",20


## 7.2 Análisis del patrón de faltantes

Antes de imputar se compara la ausencia por plan.

Si las tasas y las distribuciones cambian entre planes, utilizar una sola mediana global mezclaría comportamientos diferentes.


In [12]:
df_tiempo_revision = df.copy()

df_tiempo_revision.loc[
    mascara_tiempo_imposible,
    "monthly_watch_time_mins",
] = np.nan

faltantes_y_mediana_por_plan = (
    df_tiempo_revision
    .groupby("subscription_plan")
    .agg(
        usuarios=("user_id", "size"),
        faltantes=(
            "monthly_watch_time_mins",
            lambda serie: int(serie.isna().sum()),
        ),
        mediana_minutos=("monthly_watch_time_mins", "median"),
        media_minutos=("monthly_watch_time_mins", "mean"),
    )
)

faltantes_y_mediana_por_plan["porcentaje_faltante"] = (
    faltantes_y_mediana_por_plan["faltantes"]
    / faltantes_y_mediana_por_plan["usuarios"]
    * 100
)

display(faltantes_y_mediana_por_plan.round(2))

,usuarios,faltantes,mediana_minutos,media_minutos,porcentaje_faltante
subscription_plan,,,,,
Básico,3600,39,552.70,597.48,1.08
Estándar,2817,64,840.00,872.44,2.27
Premium,1583,170,"1,127.00","1,139.92",10.74


### Interpretación

Los planes presentan medianas claramente distintas:

- Básico: alrededor de 553 minutos;
- Estándar: alrededor de 840 minutos;
- Premium: alrededor de 1.127 minutos.

Además, la proporción de faltantes es mayor en Premium.

La ausencia puede describirse como compatible con un mecanismo **MAR** respecto del plan: la probabilidad de falta está asociada con una variable observada. No se puede demostrar el mecanismo verdadero sin información externa, pero sí existe evidencia para evitar una imputación global.

### Decisión

1. Convertir los 80 valores físicamente imposibles en faltantes.
2. Imputar todos los faltantes de minutos con la mediana del plan correspondiente.


In [13]:
cantidad_tiempos_imposibles = int(mascara_tiempo_imposible.sum())

df.loc[
    mascara_tiempo_imposible,
    "monthly_watch_time_mins",
] = np.nan

auditar(
    df,
    5,
    "Marcar tiempos mensuales imposibles como faltantes",
)

medianas_tiempo_por_plan = (
    df.groupby("subscription_plan")[
        "monthly_watch_time_mins"
    ].median()
)

df["monthly_watch_time_mins"] = (
    df["monthly_watch_time_mins"]
    .fillna(
        df["subscription_plan"].map(
            medianas_tiempo_por_plan
        )
    )
)

auditar(
    df,
    6,
    "Imputar minutos con mediana por plan",
)

print(f"Valores imposibles tratados: {cantidad_tiempos_imposibles}")
print("\nMedianas utilizadas:")
display(
    medianas_tiempo_por_plan
    .rename("mediana_imputacion")
    .reset_index()
)
print(
    f"Faltantes finales en minutos: "
    f"{df['monthly_watch_time_mins'].isna().sum()}"
)

Valores imposibles tratados: 80

Medianas utilizadas:


,subscription_plan,mediana_imputacion
0,Básico,552.70
1,Estándar,840.00
2,Premium,"1,127.00"


Faltantes finales en minutos: 0


## 7.3 Valores altos señalados por IQR

Después de quitar los valores físicamente imposibles, el método IQR todavía puede señalar consumos altos.

No se eliminarán automáticamente. Estos valores se encuentran por debajo del máximo físico mensual y pueden representar usuarios intensivos reales. El IQR es una herramienta de detección, no una orden de eliminación.


In [14]:
q1 = df["monthly_watch_time_mins"].quantile(0.25)
q3 = df["monthly_watch_time_mins"].quantile(0.75)
iqr = q3 - q1

limite_inferior_iqr = q1 - 1.5 * iqr
limite_superior_iqr = q3 + 1.5 * iqr

mascara_outlier_iqr = (
    (df["monthly_watch_time_mins"] < limite_inferior_iqr)
    | (df["monthly_watch_time_mins"] > limite_superior_iqr)
)

print(f"Límite inferior IQR: {limite_inferior_iqr:,.2f}")
print(f"Límite superior IQR: {limite_superior_iqr:,.2f}")
print(f"Casos señalados por IQR: {mascara_outlier_iqr.sum()}")
print(
    "Decisión: se conservan porque son físicamente posibles "
    "y pueden representar consumo intensivo real."
)

Límite inferior IQR: -340.10
Límite superior IQR: 1,897.90
Casos señalados por IQR: 122
Decisión: se conservan porque son físicamente posibles y pueden representar consumo intensivo real.


## 7.4 Impacto

- Se trataron 80 valores físicamente imposibles.
- Se imputaron los faltantes utilizando el contexto del plan.
- Se preservaron los ceros.
- Se conservaron los valores altos plausibles.

La variable queda completa sin recortar artificialmente la variabilidad real del consumo.


# 8. Tratamiento de tickets de soporte

## 8.1 Evidencia observada

La frecuencia normal se concentra entre 0 y 5 tickets. Luego aparece un salto directo hacia −1, 99 y 150.

Los valores −1, 99 y 150 se repiten como códigos aislados y no forman una continuidad con la distribución observada.


In [15]:
frecuencia_tickets_antes = (
    df["customer_support_tickets"]
    .value_counts()
    .sort_index()
    .rename_axis("tickets")
    .reset_index(name="cantidad")
)

display(frecuencia_tickets_antes)

,tickets,cantidad
0,-1,29
1,0,3563
2,1,2819
3,2,1157
4,3,280
5,4,71
6,5,14
7,99,35
8,150,32


## 8.2 Decisión

Se consideran inválidos exclusivamente los valores observados `−1`, `99` y `150`.

No se establece una regla automática del tipo “todo valor superior a 5 es error”, porque en otro dataset una persona podría tener legítimamente más de cinco consultas. Aquí la decisión se basa en los valores concretos y en la discontinuidad de esta distribución.

Después se utiliza la mediana de los tickets válidos, que es 1.


In [16]:
valores_ticket_invalidos = [-1, 99, 150]

mascara_ticket_invalido = (
    df["customer_support_tickets"]
    .isin(valores_ticket_invalidos)
)

cantidad_tickets_invalidos = int(
    mascara_ticket_invalido.sum()
)

df.loc[
    mascara_ticket_invalido,
    "customer_support_tickets",
] = np.nan

auditar(
    df,
    7,
    "Marcar tickets -1, 99 y 150 como faltantes",
)

mediana_tickets = df["customer_support_tickets"].median()

df["customer_support_tickets"] = (
    df["customer_support_tickets"]
    .fillna(mediana_tickets)
    .astype(int)
)

auditar(
    df,
    8,
    f"Imputar tickets con mediana global ({mediana_tickets:.0f})",
)

print(f"Tickets inválidos tratados: {cantidad_tickets_invalidos}")
print(f"Mediana utilizada: {mediana_tickets:.0f}")
print(
    f"Rango final: "
    f"{df['customer_support_tickets'].min()} a "
    f"{df['customer_support_tickets'].max()}"
)

Tickets inválidos tratados: 96
Mediana utilizada: 1
Rango final: 0 a 5


## 8.3 Impacto

Se corrigieron 96 códigos inválidos sin eliminar usuarios.

La variable queda en un rango de 0 a 5 tickets, coherente con la distribución válida observada.


# 9. Tratamiento de género favorito faltante

## 9.1 Evidencia observada

Existen 240 usuarios sin género favorito informado.

## 9.2 Alternativas consideradas

- **Eliminar filas:** descartaría usuarios que todavía poseen información válida en las demás columnas.
- **Imputar con la moda:** asignaría una preferencia que el usuario nunca declaró.
- **Crear una categoría explícita:** conserva al usuario y diferencia la ausencia de una preferencia real.

## 9.3 Decisión

Se utiliza la categoría **No informado**.


In [17]:
generos_faltantes_antes = int(
    df["favorite_genre"].isna().sum()
)

df["favorite_genre"] = (
    df["favorite_genre"]
    .fillna("No informado")
)

auditar(
    df,
    9,
    "Etiquetar género faltante como No informado",
)

print(
    f"Registros etiquetados como No informado: "
    f"{generos_faltantes_antes}"
)

display(
    df["favorite_genre"]
    .value_counts()
    .rename_axis("favorite_genre")
    .reset_index(name="cantidad")
)

Registros etiquetados como No informado: 240


,favorite_genre,cantidad
0,Comedia,1137
1,Drama,1115
2,Romance,1110
3,Documental,1107
4,Thriller,1104
5,Acción,1103
6,Crimen,1084
7,No informado,240


## 9.4 Impacto

Los 240 usuarios se mantienen en el dataset y la ausencia deja de confundirse con un género real.

Esta decisión mejora la transparencia: “No informado” significa falta de información, no una preferencia inferida.


# 10. Tratamiento de la fecha del último ingreso

## 10.1 Evidencia observada

La columna combina:

- `AAAA-MM-DD`;
- `AAAA/MM/DD`;
- fechas con el año al final;
- valores nulos;
- `not_available`;
- fechas imposibles;
- fechas futuras;
- formatos ambiguos como `01-06-2025`.

En una fecha como `01-06-2025`, los datos no indican si corresponde al 1 de junio o al 6 de enero. Forzar una interpretación introduciría información no respaldada por la fuente.


In [18]:
def clasificar_patron_fecha(valor):
    if pd.isna(valor):
        return "Nulo"

    texto = str(valor).strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", texto):
        return "AAAA-MM-DD"

    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", texto):
        return "AAAA/MM/DD"

    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", texto):
        return "Año al final"

    return "Otro texto"

patrones_fecha_antes = (
    df["last_login_date"]
    .map(clasificar_patron_fecha)
    .value_counts()
    .rename_axis("patron")
    .reset_index(name="cantidad")
)

display(patrones_fecha_antes)

,patron,cantidad
0,AAAA-MM-DD,7268
1,Nulo,320
2,Año al final,265
3,AAAA/MM/DD,133
4,Otro texto,14


## 10.2 Criterio de conversión conservador

Se aplica la siguiente lógica:

1. `AAAA-MM-DD` se interpreta como año-mes-día.
2. `AAAA/MM/DD` se interpreta como año/mes/día.
3. En formatos con el año al final:
   - si el primer número es mayor que 12, debe ser el día;
   - si el segundo número es mayor que 12, debe ser el día;
   - si ambos son menores o iguales a 12, el formato es ambiguo y se conserva como faltante.
4. Fechas imposibles, textos y fechas posteriores al 27 de junio de 2026 quedan como `NaT`.

No se imputan fechas porque una mediana o una moda inventaría un último acceso que no fue observado.


In [19]:
fecha_corte = pd.Timestamp("2026-06-27")

def convertir_fecha_con_estado(valor):
    if pd.isna(valor):
        return pd.NaT, "faltante_original"

    texto = str(valor).strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", texto):
        fecha = pd.to_datetime(
            texto,
            format="%Y-%m-%d",
            errors="coerce",
        )
        estado = (
            "iso_valida"
            if pd.notna(fecha)
            else "fecha_invalida"
        )
        return fecha, estado

    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", texto):
        fecha = pd.to_datetime(
            texto,
            format="%Y/%m/%d",
            errors="coerce",
        )
        estado = (
            "iso_barra_valida"
            if pd.notna(fecha)
            else "fecha_invalida"
        )
        return fecha, estado

    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", texto):
        primero, segundo, anio = map(
            int,
            texto.split("-"),
        )

        if primero > 12 and segundo <= 12:
            fecha = pd.to_datetime(
                texto,
                format="%d-%m-%Y",
                errors="coerce",
            )
            estado = (
                "dd_mm_yyyy_valida"
                if pd.notna(fecha)
                else "fecha_invalida"
            )
            return fecha, estado

        if primero <= 12 and segundo > 12:
            fecha = pd.to_datetime(
                texto,
                format="%m-%d-%Y",
                errors="coerce",
            )
            estado = (
                "mm_dd_yyyy_valida"
                if pd.notna(fecha)
                else "fecha_invalida"
            )
            return fecha, estado

        if primero <= 12 and segundo <= 12:
            return pd.NaT, "formato_ambiguo"

        return pd.NaT, "fecha_invalida"

    return pd.NaT, "texto_no_fecha"

resultados_fecha = (
    df["last_login_date"]
    .map(convertir_fecha_con_estado)
)

fechas_convertidas = pd.Series(
    [resultado[0] for resultado in resultados_fecha],
    index=df.index,
    dtype="datetime64[ns]",
)

estado_fecha = pd.Series(
    [resultado[1] for resultado in resultados_fecha],
    index=df.index,
    dtype="object",
)

mascara_fecha_futura = fechas_convertidas > fecha_corte

estado_fecha.loc[mascara_fecha_futura] = "fecha_futura"
fechas_convertidas.loc[mascara_fecha_futura] = pd.NaT

df["last_login_date"] = fechas_convertidas

auditar(
    df,
    10,
    (
        "Convertir fechas válidas y conservar como "
        "faltantes las ambiguas, inválidas o futuras"
    ),
)

resumen_conversion_fechas = (
    estado_fecha
    .value_counts()
    .rename_axis("estado")
    .reset_index(name="cantidad")
)

display(resumen_conversion_fechas)

print(
    f"Fechas válidas finales: "
    f"{df['last_login_date'].notna().sum()}"
)
print(
    f"Fechas faltantes finales: "
    f"{df['last_login_date'].isna().sum()}"
)

,estado,cantidad
0,iso_valida,7216
1,faltante_original,320
2,iso_barra_valida,133
3,formato_ambiguo,102
4,mm_dd_yyyy_valida,78
5,dd_mm_yyyy_valida,72
6,fecha_invalida,50
7,fecha_futura,15
8,texto_no_fecha,14


Fechas válidas finales: 7499
Fechas faltantes finales: 501


## 10.3 Impacto

El dataset final conserva 7.499 fechas válidas y 501 faltantes.

Que todavía existan faltantes no representa un fallo del proceso. En este caso es una decisión de calidad: se prefiere reconocer que la fecha es desconocida antes que asignar una fecha potencialmente incorrecta.

Los análisis posteriores que utilicen esta variable deberán trabajar con los casos válidos o documentar un tratamiento específico.


# 11. Validación final del dataset

Se aplican controles explícitos para comprobar que:

- cada usuario aparezca una sola vez;
- las categorías pertenezcan a los conjuntos esperados;
- las variables numéricas respeten los rangos definidos;
- los faltantes remanentes estén concentrados únicamente en la fecha;
- el archivo original continúe intacto.


In [20]:
planes_validos = {"Básico", "Estándar", "Premium"}

paises_validos = {
    "Argentina",
    "Brasil",
    "Chile",
    "Colombia",
    "México",
    "Perú",
    "Uruguay",
}

generos_validos = {
    "Acción",
    "Comedia",
    "Crimen",
    "Documental",
    "Drama",
    "Romance",
    "Thriller",
    "No informado",
}

controles = {
    "8.000 usuarios finales": len(df) == 8000,
    "user_id sin duplicados": not df["user_id"].duplicated().any(),
    "Edad entre 13 y 100": df["age"].between(13, 100).all(),
    "Minutos entre 0 y 44.640": (
        df["monthly_watch_time_mins"]
        .between(0, minutos_mes_31_dias)
        .all()
    ),
    "Tickets entre 0 y 5": (
        df["customer_support_tickets"]
        .between(0, 5)
        .all()
    ),
    "Planes válidos": (
        set(df["subscription_plan"].unique())
        == planes_validos
    ),
    "Países válidos": (
        set(df["country"].unique())
        == paises_validos
    ),
    "Géneros válidos": (
        set(df["favorite_genre"].unique())
        == generos_validos
    ),
    "Solo fecha conserva nulos": (
        df.drop(columns="last_login_date")
        .isna()
        .sum()
        .sum()
        == 0
    ),
    "Dataset original sin modificaciones": (
        df_raw.equals(pd.read_json(ruta_datos))
    ),
}

tabla_controles = pd.DataFrame({
    "control": list(controles.keys()),
    "resultado": list(controles.values()),
})

display(tabla_controles)

if not all(controles.values()):
    controles_fallidos = [
        nombre
        for nombre, resultado in controles.items()
        if not resultado
    ]
    raise AssertionError(
        f"Fallaron los controles: {controles_fallidos}"
    )

print("Todos los controles de calidad fueron superados.")

,control,resultado
0,8.000 usuarios finales,True
1,user_id sin duplicados,True
2,Edad entre 13 y 100,True
3,Minutos entre 0 y 44.640,True
4,Tickets entre 0 y 5,True
5,Planes válidos,True
6,Países válidos,True
7,Géneros válidos,True
8,Solo fecha conserva nulos,True
9,Dataset original sin modificaciones,True


Todos los controles de calidad fueron superados.


## 11.1 Comparación inicial y final


In [21]:
comparacion_inicial_final = pd.DataFrame({
    "indicador": [
        "Filas",
        "Columnas",
        "Identificadores duplicados",
        "Nulos totales",
        "Categorías de plan",
        "Categorías de país",
        "Categorías de género incluyendo faltantes",
        "Edad mínima",
        "Edad máxima",
        "Minutos mínimos",
        "Minutos máximos",
        "Tickets mínimos",
        "Tickets máximos",
    ],
    "inicial": [
        len(df_raw),
        df_raw.shape[1],
        int(df_raw.duplicated(subset="user_id").sum()),
        int(df_raw.isna().sum().sum()),
        df_raw["subscription_plan"].nunique(dropna=False),
        df_raw["country"].nunique(dropna=False),
        df_raw["favorite_genre"].nunique(dropna=False),
        df_raw["age"].min(),
        df_raw["age"].max(),
        df_raw["monthly_watch_time_mins"].min(),
        df_raw["monthly_watch_time_mins"].max(),
        df_raw["customer_support_tickets"].min(),
        df_raw["customer_support_tickets"].max(),
    ],
    "final": [
        len(df),
        df.shape[1],
        int(df.duplicated(subset="user_id").sum()),
        int(df.isna().sum().sum()),
        df["subscription_plan"].nunique(dropna=False),
        df["country"].nunique(dropna=False),
        df["favorite_genre"].nunique(dropna=False),
        df["age"].min(),
        df["age"].max(),
        df["monthly_watch_time_mins"].min(),
        df["monthly_watch_time_mins"].max(),
        df["customer_support_tickets"].min(),
        df["customer_support_tickets"].max(),
    ],
})

display(comparacion_inicial_final)

,indicador,inicial,final
0,Filas,"8,160.00","8,000.00"
1,Columnas,8.00,8.00
2,Identificadores duplicados,160.00,0.00
3,Nulos totales,753.00,501.00
4,Categorías de plan,15.00,3.00
5,Categorías de país,26.00,7.00
6,Categorías de género incluyendo faltantes,29.00,8.00
7,Edad mínima,-5.00,13.00
8,Edad máxima,150.00,80.00
9,Minutos mínimos,-120.00,0.00


# 12. Guardado del dataset procesado y del log ETL

El dataset limpio se guarda en formato CSV porque puede ser utilizado fácilmente desde pandas, Streamlit y otros entornos.

Las fechas se exportan como `AAAA-MM-DD`. Los faltantes de fecha se conservan como celdas vacías.

También se guarda el registro completo del pipeline con las columnas mínimas solicitadas por la consigna.


In [22]:
rutas_salida_datos = [
    Path("../data/processed/streaming_users_clean.csv"),
    Path("data/processed/streaming_users_clean.csv"),
]

rutas_salida_log = [
    Path("../logs/pipeline_log.csv"),
    Path("logs/pipeline_log.csv"),
]

ruta_salida_datos = next(
    (
        ruta for ruta in rutas_salida_datos
        if ruta.parent.exists()
    ),
    rutas_salida_datos[0],
)

ruta_salida_log = next(
    (
        ruta for ruta in rutas_salida_log
        if ruta.parent.exists()
    ),
    rutas_salida_log[0],
)

df.to_csv(
    ruta_salida_datos,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

log_df = pd.DataFrame(pipeline_log)

log_df.to_csv(
    ruta_salida_log,
    index=False,
    encoding="utf-8-sig",
)

print(f"Dataset procesado guardado en: {ruta_salida_datos.resolve()}")
print(f"Log ETL guardado en: {ruta_salida_log.resolve()}")

print("\nLog completo:")
display(log_df)

Dataset procesado guardado en: /mnt/data/PI_Mineria_Datos_1_GRUPAL/data/processed/streaming_users_clean.csv
Log ETL guardado en: /mnt/data/PI_Mineria_Datos_1_GRUPAL/logs/pipeline_log.csv

Log completo:


,Paso,Descripción,Filas,Nulos,Retención (%)
0,0,Dataset original,8160,753,100.00
1,1,Conservar la primera aparición de cada user_id,8000,753,98.04
2,2,"Estandarizar categorías de plan, país y género",8000,753,98.04
3,3,Marcar edades fuera del rango 13-100 como falt...,8000,873,98.04
4,4,Imputar edad con mediana global (33),8000,753,98.04
5,5,Marcar tiempos mensuales imposibles como falta...,8000,833,98.04
6,6,Imputar minutos con mediana por plan,8000,560,98.04
7,7,"Marcar tickets -1, 99 y 150 como faltantes",8000,656,98.04
8,8,Imputar tickets con mediana global (1),8000,560,98.04
9,9,Etiquetar género faltante como No informado,8000,320,98.04


# 13. Verificación de los archivos guardados

Los archivos se vuelven a leer para comprobar que:

- existen;
- tienen la cantidad esperada de filas;
- el log contiene todos los pasos;
- los tipos principales pueden recuperarse correctamente.


In [23]:
df_verificacion = pd.read_csv(
    ruta_salida_datos,
    parse_dates=["last_login_date"],
)

log_verificacion = pd.read_csv(ruta_salida_log)

print(f"Dataset recuperado: {df_verificacion.shape}")
print(f"Pasos recuperados del log: {len(log_verificacion)}")
print(
    f"Duplicados recuperados: "
    f"{df_verificacion['user_id'].duplicated().sum()}"
)
print(
    f"Nulos recuperados en fecha: "
    f"{df_verificacion['last_login_date'].isna().sum()}"
)

assert df_verificacion.shape == (8000, 8)
assert df_verificacion["user_id"].duplicated().sum() == 0
assert len(log_verificacion) == len(pipeline_log)

print("Los archivos fueron guardados y recuperados correctamente.")

Dataset recuperado: (8000, 8)
Pasos recuperados del log: 11
Duplicados recuperados: 0
Nulos recuperados en fecha: 501
Los archivos fueron guardados y recuperados correctamente.


# 14. Conclusión del proceso de limpieza

El dataset original tenía 8.160 filas y el dataset final conserva 8.000 usuarios únicos, lo que representa una retención del 98,04 %.

Las únicas filas eliminadas fueron 160 apariciones posteriores de identificadores que ya estaban presentes. Las demás correcciones conservaron a los usuarios:

- se normalizaron categorías;
- se imputaron edades inválidas con la mediana global;
- se imputaron minutos faltantes o imposibles con la mediana de cada plan;
- se imputaron códigos inválidos de tickets con la mediana;
- se creó la categoría `No informado` para preferencias ausentes;
- se convirtieron fechas válidas y se mantuvieron como faltantes las fechas que no podían interpretarse sin adivinar.

El dataset final todavía contiene 501 valores faltantes en `last_login_date`. Esta ausencia es intencional y documentada: representa casos en los que no existe evidencia suficiente para reconstruir la fecha.